# Customer Churn Prediction — Exploratory Data Analysis

This notebook loads the public IBM Telco Customer Churn dataset, validates data quality, explores churn patterns, and prepares evidence for the machine-learning stage.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.download_data import download_dataset
from src.data_preprocessing import load_data, clean_data

pd.set_option('display.max_columns', None)
download_dataset()

## 1. Load and inspect the dataset

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
df_raw = load_data(DATA_PATH)
print(f'Rows: {df_raw.shape[0]:,}')
print(f'Columns: {df_raw.shape[1]:,}')
display(df_raw.head())

In [ ]:
df_raw.info()
print('\nDuplicate rows:', df_raw.duplicated().sum())
print('\nMissing values:')
display(df_raw.isna().sum().sort_values(ascending=False).to_frame('missing'))

## 2. Clean the data

`TotalCharges` contains blank values in the raw dataset. The preprocessing module converts it to numeric and removes rows where that conversion produces missing values. Duplicate rows are also removed.

In [ ]:
df = clean_data(df_raw)
print(f'Rows after cleaning: {df.shape[0]:,}')
print(f'Rows removed: {len(df_raw) - len(df):,}')
print('Remaining missing values:', int(df.isna().sum().sum()))

## 3. Churn distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True).mul(100).round(2)
display(pd.DataFrame({'Customers': churn_counts, 'Percentage': churn_pct}))

ax = churn_counts.plot(kind='bar', title='Customer Churn Distribution')
ax.set_xlabel('Churn')
ax.set_ylabel('Customers')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Churn by contract type

In [ ]:
contract_churn = (df.groupby('Contract')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).sort_values(ascending=False))
display(contract_churn.round(2).to_frame('Churn Rate (%)'))

contract_churn.plot(kind='bar', title='Churn Rate by Contract Type')
plt.ylabel('Churn Rate (%)')
plt.xlabel('Contract Type')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Churn by internet service and payment method

In [ ]:
def churn_rate_by(column):
    return (df.groupby(column)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).sort_values(ascending=False).round(2))

for column in ['InternetService', 'PaymentMethod']:
    print(f'\n{column}')
    display(churn_rate_by(column).to_frame('Churn Rate (%)'))

## 6. Numerical relationships

In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
display(df.groupby('Churn')[numeric_cols].mean().round(2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, column in zip(axes, numeric_cols):
    df.boxplot(column=column, by='Churn', ax=ax)
    ax.set_title(column)
    ax.set_xlabel('Churn')
    ax.set_ylabel(column)
fig.suptitle('Numerical Features by Churn Status')
plt.tight_layout()
plt.show()

## 7. Churn by senior-citizen status

In [ ]:
senior_churn = churn_rate_by('SeniorCitizen')
senior_churn.index = senior_churn.index.map({0: 'Non-Senior', 1: 'Senior'})
display(senior_churn.to_frame('Churn Rate (%)'))

## 8. Key observations

Use the verified outputs above to summarize 3–5 findings. The final project README will use these evidence-based observations rather than generic claims.

In [ ]:
print('EDA complete. Review the tables and charts above before writing the final business conclusions.')